In [0]:
transactions_processed_df = spark.read.table("processed_db.transactions")


In [0]:
customers_processed_df = spark.read.table("processed_db.customers")
       

###Join Customers & Transactions

In [0]:

customers_df_renamed = customers_processed_df.withColumnRenamed(
    "processed_timestamp", "customers_processed_ts"
)

transactions_df_renamed = transactions_processed_df.withColumnRenamed(
    "processed_timestamp", "transactions_processed_ts"
)


customer_transactions_df = (
    customers_df_renamed.alias("c")
    .join(
        transactions_df_renamed.alias("t"),
        on="customer_id",
        how="left"
    )
)

###Total orders per customer, Total spend, Average order value, First & last purchase

In [0]:
from pyspark.sql.functions import (
    col, countDistinct, sum, avg, min, max
)

customer_metrics_df = (
    customer_transactions_df
        .groupBy(
            col("customer_id"),
            col("name"),
            col("city"),
            col("age_group")
        )
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum("amount").alias("total_spend"),
            avg("amount").alias("avg_order_value"),
            min("order_date").alias("first_order_date"),
            max("order_date").alias("last_order_date")
        )
        .orderBy(col("total_spend").desc())
)

customer_metrics_df.show()

+-----------+------+---------+---------+------------+-----------+------------------+----------------+---------------+
|customer_id|  name|     city|age_group|total_orders|total_spend|   avg_order_value|first_order_date|last_order_date|
+-----------+------+---------+---------+------------+-----------+------------------+----------------+---------------+
|       C002| Anita|   Mumbai|    26-35|           5|   161500.0|           32300.0|      2024-01-02|     2024-03-18|
|       C005|  Amit|     Pune|    36-45|           2|   106000.0|           53000.0|      2024-02-03|     2024-03-20|
|       C003|Suresh|    Delhi|    36-45|           3|    54500.0|18166.666666666668|      2024-01-10|     2024-03-25|
|       C009| Manoj|    Delhi|      46+|           1|    45000.0|           45000.0|      2024-03-10|     2024-03-10|
|       C007|  Ravi|     NULL|    26-35|           1|    24000.0|           24000.0|      2024-02-22|     2024-02-22|
|       C001| Rahul|Bangalore|    26-35|           4|   

###Customer segmentation by spend

In [0]:
from pyspark.sql.functions import when

customer_segment_df = (
    customer_metrics_df
        .withColumn(
            "customer_segment",
            when(col("total_spend") >= 100000, "High Value")
            .when(col("total_spend") >= 30000, "Medium Value")
            .when(col("total_spend").isNull(), "No Purchase")
            .otherwise("Low Value")
        )
)

customer_segment_df.show()

+-----------+------+---------+---------+------------+-----------+------------------+----------------+---------------+----------------+
|customer_id|  name|     city|age_group|total_orders|total_spend|   avg_order_value|first_order_date|last_order_date|customer_segment|
+-----------+------+---------+---------+------------+-----------+------------------+----------------+---------------+----------------+
|       C002| Anita|   Mumbai|    26-35|           5|   161500.0|           32300.0|      2024-01-02|     2024-03-18|      High Value|
|       C005|  Amit|     Pune|    36-45|           2|   106000.0|           53000.0|      2024-02-03|     2024-03-20|      High Value|
|       C003|Suresh|    Delhi|    36-45|           3|    54500.0|18166.666666666668|      2024-01-10|     2024-03-25|    Medium Value|
|       C009| Manoj|    Delhi|      46+|           1|    45000.0|           45000.0|      2024-03-10|     2024-03-10|    Medium Value|
|       C007|  Ravi|     NULL|    26-35|           1|  

###Revenue by city

In [0]:
revenue_by_city_df = (
    customer_transactions_df
        .groupBy("city")
        .agg(
            countDistinct("customer_id").alias("total_customers"),
            sum("amount").alias("total_revenue")
        )
        .orderBy(col("total_revenue").desc())
)

revenue_by_city_df.show()

+---------+---------------+-------------+
|     city|total_customers|total_revenue|
+---------+---------------+-------------+
|   Mumbai|              2|     177500.0|
|     Pune|              1|     106000.0|
|    Delhi|              2|      99500.0|
|     NULL|              1|      24000.0|
|Bangalore|              2|      22700.0|
|  Kolkata|              1|      17000.0|
|  Chennai|              1|      12000.0|
+---------+---------------+-------------+



###Revenue by age group

In [0]:
revenue_by_age_group_df = (
    customer_transactions_df
        .groupBy("age_group")
        .agg(
            countDistinct("customer_id").alias("customers"),
            sum("amount").alias("total_revenue")
        )
        .orderBy(col("total_revenue").desc())
)

revenue_by_age_group_df.show()

+---------+---------+-------------+
|age_group|customers|total_revenue|
+---------+---------+-------------+
|    26-35|        5|     234700.0|
|    36-45|        3|     177500.0|
|      46+|        1|      45000.0|
|  Unknown|        1|       1500.0|
+---------+---------+-------------+



###Daily sales trend

In [0]:
from pyspark.sql.functions import to_date

daily_sales_df = (
    transactions_processed_df
        .groupBy(to_date(col("order_date")).alias("order_date"))
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum("quantity").alias("total_quantity"),
            sum("amount").alias("total_revenue")
        )
        .orderBy("order_date")
)

daily_sales_df.show()

+----------+------------+--------------+-------------+
|order_date|total_orders|total_quantity|total_revenue|
+----------+------------+--------------+-------------+
|2024-01-01|           1|             1|      15000.0|
|2024-01-02|           1|             1|      55000.0|
|2024-01-05|           1|             2|       2000.0|
|2024-01-10|           1|             1|      16000.0|
|2024-01-15|           1|             1|      25000.0|
|2024-02-01|           1|             1|      12000.0|
|2024-02-03|           1|             1|      52000.0|
|2024-02-10|           1|             1|      18000.0|
|2024-02-15|           1|             2|       3000.0|
|2024-02-18|           1|             1|      15500.0|
|2024-02-20|           1|             1|       1500.0|
|2024-02-22|           1|             1|      24000.0|
|2024-03-01|           1|             1|      17000.0|
|2024-03-05|           1|             1|      60000.0|
|2024-03-10|           1|             1|      45000.0|
|2024-03-1

###Top customers by revenue

In [0]:
top_customers_df = (
    customer_metrics_df
        .select("customer_id", "name", "total_spend")
        .orderBy(col("total_spend").desc())
        .limit(10)
)

top_customers_df.show()

+-----------+------+-----------+
|customer_id|  name|total_spend|
+-----------+------+-----------+
|       C002| Anita|   161500.0|
|       C005|  Amit|   106000.0|
|       C003|Suresh|    54500.0|
|       C009| Manoj|    45000.0|
|       C007|  Ravi|    24000.0|
|       C001| Rahul|    21200.0|
|       C008|Kavita|    17000.0|
|       C010|Sunita|    16000.0|
|       C004| Priya|    12000.0|
|       C006|  Neha|     1500.0|
+-----------+------+-----------+



###Customers with no transactions

In [0]:
customers_no_txn_df = (
    customer_metrics_df
        .filter(col("total_orders") == 0)
)

customers_no_txn_df.show()

+-----------+----+----+---------+------------+-----------+---------------+----------------+---------------+
|customer_id|name|city|age_group|total_orders|total_spend|avg_order_value|first_order_date|last_order_date|
+-----------+----+----+---------+------------+-----------+---------------+----------------+---------------+
+-----------+----+----+---------+------------+-----------+---------------+----------------+---------------+



###Save the output to presentation container as delta table

In [0]:
customer_transactions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customer_transactions")

In [0]:
customer_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customer_metrics")

In [0]:
customer_segment_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customer_segment")

In [0]:
revenue_by_city_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.revenue_by_city")

In [0]:
revenue_by_age_group_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.revenue_by_age_group")

In [0]:
daily_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.daily_sales")

In [0]:
top_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.top_customers")

In [0]:
customers_no_txn_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customers_no_txn")